In [ ]:
import spikeinterface.full as si
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# PATHS (set here when running standalone; overridden by main_pipeline)
try:
    openephys_folder
    base_folder
except NameError:
    openephys_folder = Path(r'C:\path\to\openephys\session_folder')
    base_folder      = Path(r'C:\Users\labuser\Ilaria\Project\processed\openephys_output')

In [ ]:
# OPENEPHYS-SPECIFIC PARAMETERS
stream_name = 'Signals CH'
# run this to list available streams:
# print(si.get_neo_streams('openephys', openephys_folder))

freq_min       = 300.    # highpass cutoff (Hz)
cref_operator  = 'median'
cref_reference = 'global'

# SORTER (set in main_pipeline; fallback when running standalone)
try:
    sorter = SORTER_KS
except NameError:
    sorter = 'kilosort4'   # 'kilosort2_5' | 'kilosort4'

# ── OPTIONAL PREPROCESSING FEATURES ─────────────────────────────────────────
# Manual channel selection — list of channel IDs to keep; None → keep all.
# Example: manual_channel_ids = ['CH1', 'CH2', 'CH3', 'CH4']
manual_channel_ids = None

# Extra bad channels — force-remove on top of auto-detected ones.
extra_bad_channel_ids = []

# Artifact removal — timestamps (seconds) to blank; leave empty to skip.
artifact_timestamps_s = []   # e.g. [10.5, 23.7, 45.2]
artifact_ms_before    = 2.0
artifact_ms_after     = 5.0

# TTL-triggered artifact removal — reads digital events from the recording and
# blanks a window around each pulse.  False → skip.
use_ttl_artifacts = False
ttl_channel_id    = 'Rhythm FPGA TTL Input'   # adjust to match your recording

In [ ]:
# LOAD
raw_rec = si.read_openephys(openephys_folder, stream_name=stream_name, load_sync_channel=False)
print(raw_rec)

In [ ]:
# TRACES VISUALIZATION
_fs = raw_rec.get_sampling_frequency()
raw_2s = raw_rec.frame_slice(0, int(2 * _fs))

si.plot_traces(raw_2s, channel_ids=raw_rec.channel_ids[:10], mode='line', backend='matplotlib')
plt.suptitle('Raw traces')
plt.tight_layout()
plt.show()

In [ ]:
# PREPROCESSING

# Optional: manual channel selection
if manual_channel_ids is not None:
    raw_rec = raw_rec.channel_slice(manual_channel_ids)
    print(f"Manual selection: kept {len(manual_channel_ids)} channels")

rec1 = si.highpass_filter(raw_rec, freq_min=freq_min)

# Optional: artifact removal (TTL events + manual timestamps)
_artifact_frames = []
if use_ttl_artifacts:
    _events  = si.read_openephys_event(openephys_folder, block_index=0)
    _ttl_s   = _events.get_event_times(channel_id=ttl_channel_id, segment_index=0)
    _t0      = raw_rec.get_times(segment_index=0)[0]
    _fs      = raw_rec.get_sampling_frequency()
    _artifact_frames.extend(((_ttl_s - _t0) * _fs).astype(np.int64).tolist())
    print(f"TTL artifacts: {len(_ttl_s)} event(s) found")
if artifact_timestamps_s:
    _fs = raw_rec.get_sampling_frequency()
    _artifact_frames.extend([int(t * _fs) for t in artifact_timestamps_s])
if _artifact_frames:
    rec1 = si.remove_artifacts(
        rec1,
        list_triggers=[np.array(sorted(_artifact_frames), dtype=np.int64)],
        ms_before=artifact_ms_before, ms_after=artifact_ms_after,
        mode='zeros',
    )
    print(f"Artifact removal: blanked {len(_artifact_frames)} event(s) total")

bad_channel_ids, _ = si.detect_bad_channels(rec1)

# Optional: extra bad channels
if extra_bad_channel_ids:
    bad_channel_ids = list(set(bad_channel_ids.tolist()) | set(extra_bad_channel_ids))

rec2 = rec1.remove_channels(bad_channel_ids)
rec  = si.common_reference(rec2, operator=cref_operator, reference=cref_reference)
print(rec)

In [ ]:
# SPIKE SORTING
import gc, shutil, json

job_kwargs = dict(n_jobs=4, chunk_duration='1s', progress_bar=True)

_preprocess_dir = base_folder / 'preprocess'
_sorter_dir     = base_folder / f'{sorter}_output'

try:
    del analyzer
except NameError:
    pass
gc.collect()

def _si_folder_valid(folder):
    """True if folder contains a complete SpikeInterface saved object."""
    if not folder.exists():
        return False
    if any((folder / f).exists() for f in ('si_folder.json', 'cached.json', 'cached.pkl', 'cached.pickle')):
        return True
    if ((folder / 'sorter_output').is_dir()
            and (folder / 'spikeinterface_params.json').exists()
            and (folder / 'spikeinterface_log.json').exists()):
        with open(folder / 'spikeinterface_log.json') as _f:
            _log = json.load(_f)
        return not _log.get('error', False)
    return False

def _safe_rmtree(folder):
    try:
        shutil.rmtree(folder)
    except PermissionError:
        raise RuntimeError(
            f"Cannot delete '{folder.name}' — a file is still locked by another process.\n"
            "If the spikeinterface_gui is open, close it.\n"
            "Then restart the Jupyter kernel and re-run."
        )

if _si_folder_valid(_preprocess_dir):
    rec = si.load(_preprocess_dir)
    print(f"Loaded cached preprocessed recording from '{_preprocess_dir.name}'")
else:
    if _preprocess_dir.exists():
        _safe_rmtree(_preprocess_dir)
    rec = rec.save(folder=_preprocess_dir, format='binary', **job_kwargs)

if _si_folder_valid(_sorter_dir):
    sorting = si.load(_sorter_dir)
    print(f"Loaded cached sorting from '{_sorter_dir.name}'")
else:
    if _sorter_dir.exists():
        _safe_rmtree(_sorter_dir)
    sorting = si.run_sorter(sorter, rec, folder=_sorter_dir, verbose=True)

print(sorting)

In [ ]:
# ANALYSIS & UNIT REFINEMENT
import sys, subprocess, pandas as pd
import spikeinterface.curation as sc
import spikeinterface.widgets as sw

# auto-install dependencies if missing
for _pkg, _import in [("huggingface_hub", "huggingface_hub"), ("skops", "skops")]:
    try:
        __import__(_import)
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", _pkg], check=True)

# picks up flag from main_pipeline; defaults to True when running standalone
try:
    _run_ur = RUN_UNITREFINE
except NameError:
    _run_ur = True

# Reload from disk to get a clean file handle after the overwrite — required on Windows
rec_analysis = si.load(base_folder / 'preprocess')

analyzer = si.create_sorting_analyzer(sorting=sorting, recording=rec_analysis, format="memory")
analyzer.compute([
    "random_spikes",
    "waveforms",
    "templates",
    "noise_levels",
    "spike_amplitudes",
    "principal_components",
    "spike_locations",
    "unit_locations",
    "quality_metrics",
])
analyzer.compute("template_metrics", include_multi_channel_metrics=True)

if _run_ur:
    noise_df = sc.auto_label_units(
        sorting_analyzer=analyzer,
        repo_id="SpikeInterface/UnitRefine_noise_neural_classifier",
        trust_model=True,
    )
    sua_mua_df = sc.auto_label_units(
        sorting_analyzer=analyzer,
        repo_id="SpikeInterface/UnitRefine_sua_mua_classifier",
        trust_model=True,
    )

    final_labels = noise_df["prediction"].copy()
    neural_mask  = final_labels == "neural"
    final_labels[neural_mask] = sua_mua_df["prediction"][neural_mask]

    print(final_labels.value_counts())
    print(f"\nSUA  : {(final_labels == 'sua').sum()}")
    print(f"MUA  : {(final_labels == 'mua').sum()}")
    print(f"Noise: {(final_labels == 'noise').sum()}")

    analyzer.sorting.set_property("quality", [final_labels[uid] for uid in analyzer.sorting.unit_ids])

    final_labels.rename("unitrefine_label").to_csv(base_folder / 'curation_labels.csv', header=True)
    good_unit_ids = final_labels[final_labels == "sua"].index.tolist()
    print(f"\nGood (SUA) units saved to curation_labels.csv")

In [ ]:
# QUALITY METRICS
qm = analyzer.get_extension("quality_metrics").get_data()
print(qm)
qm.to_csv(base_folder / 'quality_metrics.csv')
print(f"Quality metrics saved → {base_folder / 'quality_metrics.csv'}")

In [ ]:
# INTERACTIVE VIEWER (sigui)
# shows waveforms, ISI, amplitudes, probe map — click any unit to inspect
import spikeinterface_gui
spikeinterface_gui.run_mainwindow(analyzer)